# VIscaNEr — Candidate Transformer vs Fusion MLP

Freezes both Stage-2C DINOv3-B branches, recomputes their candidate features, and trains a compact permutation-equivariant Transformer on `hard_train`. `hard_val` selects the checkpoint; all test splits are evaluated only after model selection.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/VIscaNEr_transformer')
OUTPUT = Path('/kaggle/working/fusion_transformer')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'

def dataset_root(slug, marker):
    roots = set()
    for match in INPUT.rglob(marker):
        for parent in (match, *match.parents):
            if parent.name == slug:
                roots.add(parent); break
    if len(roots) != 1:
        raise RuntimeError(f'Attach exactly one dataset {slug} containing {marker}; found: {sorted(roots)}')
    return roots.pop()

def stage1_checkpoint(kind):
    token = f'dinov3_vitb16_{kind}_hard_finetune'
    matches = [path for path in INPUT.rglob('best.pt') if token in path.as_posix()]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one Stage-I {kind} checkpoint; found: {matches}')
    return matches[0]

print('SETUP | resolving code and datasets...', flush=True)
code = dataset_root('viscaner-fusion-stage2-code', 'code_sha256.json')
hard = dataset_root('viscaner-fusion-hardset-v1', 'manifest.csv')
labels = dataset_root('viscaner-dinov3-data', 'crops_metadata.csv')
bottles = dataset_root('viscaner-bottle-classifier-data', 'audit_metadata.csv')
label_checkpoint = stage1_checkpoint('labels')
bottle_checkpoint = stage1_checkpoint('bottles')
shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(code, PROJECT, ignore=shutil.ignore_patterns('dataset-metadata.json'))
for archive in PROJECT.glob('*.zip'):
    destination = PROJECT / archive.stem
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(destination)
checksums = json.loads((PROJECT/'code_sha256.json').read_text())
for name, expected in checksums.items():
    actual = hashlib.sha256((PROJECT/name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Code checksum mismatch: {name}')
sys.path.insert(0, str(PROJECT))
from fusion_stage2.kaggle_io import materialize_dataset_dirs
cache = Path('/kaggle/working/dataset_cache')
hard = materialize_dataset_dirs(hard, ('label_crops', 'bottle_crops'), cache)
labels = materialize_dataset_dirs(labels, ('refs', 'crops'), cache)
bottles = materialize_dataset_dirs(bottles, ('refs', 'crops'), cache)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=5.6,<6', 'safetensors>=0.5,<1', 'rapidfuzz>=3,<4', 'Unidecode>=1.3,<2'], check=True)
import torch

def adapted_checkpoint():
    required = {'fusion_state_dict', 'label_model_state_dict', 'bottle_model_state_dict', 'manual_val', 'hard_val'}
    matches = []
    for path in INPUT.rglob('*_best.pt'):
        try:
            payload = torch.load(path, map_location='cpu', weights_only=False, mmap=True)
            if isinstance(payload, dict) and required.issubset(payload):
                matches.append(path)
            del payload
        except Exception:
            continue
    if len(matches) != 1:
        raise RuntimeError(f'Expected one completed Stage-2C checkpoint; found: {matches}')
    return matches[0]

adapted = adapted_checkpoint()
required = (hard/'manifest.csv', hard/'label_crops', hard/'bottle_crops', labels/'refs', labels/'crops', bottles/'refs', bottles/'crops', label_checkpoint, bottle_checkpoint, adapted)
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU accelerator before running this notebook')
print(f'SETUP | verified {len(checksums)} code files | GPU={torch.cuda.get_device_name(0)}', flush=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', '-m', 'fusion_stage2.train_transformer_ranker',
    '--manifest', str(hard/'manifest.csv'), '--hard-root', str(hard),
    '--label-data-root', str(labels), '--bottle-data-root', str(bottles),
    '--label-refs-root', str(labels/'refs'), '--bottle-refs-root', str(bottles/'refs'),
    '--label-base-checkpoint', str(label_checkpoint), '--bottle-base-checkpoint', str(bottle_checkpoint),
    '--adapted-checkpoint', str(adapted), '--output-dir', str(OUTPUT), '--device', 'cuda',
    '--top-k', '10', '--epochs', '40', '--min-epochs', '8', '--patience', '7',
    '--batch-size', '256', '--eval-batch-size', '512', '--embedding-batch-size', '48',
    '--d-model', '96', '--num-heads', '4', '--num-layers', '2', '--feedforward-dim', '256',
    '--learning-rate', '2e-4', '--pairwise-weight', '0.15',
]
print('TRAINING | frozen DINO-B x2; Candidate Transformer vs frozen Fusion MLP', flush=True)
subprocess.run(command, cwd=PROJECT, check=True)
summary = json.loads((OUTPUT/'transformer_final_metrics.json').read_text())
print('FINAL TRANSFORMER EXPERIMENT |', json.dumps(summary, ensure_ascii=False), flush=True)
